# 🧬 Embeddings Deep Dive

Embeddings turn text into fixed-length vectors of floats. Text that means similar
things lands in nearby regions of that vector space, which is what makes semantic
search, clustering and retrieval-augmented generation possible.

This notebook works through four practical skills: embedding one text, embedding
many at once, ranking documents by similarity, and caching so you stop paying for
vectors you have already computed.

## Learning Objectives
In this notebook, you will learn:
1. **Single embeddings** - use `embed_query()` and read the vector's shape and norm
2. **Batch embeddings** - use `embed_documents()` to embed a list in one request
3. **Cosine similarity** - rank documents against a query by vector closeness
4. **Embedding caching** - wrap a model with `CacheBackedEmbeddings` to skip repeat API calls

## Prerequisites
- An `OPENAI_API_KEY` in a `.env` file at the repo root
- `langchain-openai`, `langchain-classic`, `numpy`, `python-dotenv`
- Completed: `03_embeddings.ipynb`

---

## 🔧 1. Environment Setup

We load the API key from `.env` and build one `OpenAIEmbeddings` client that every
section reuses. `text-embedding-3-small` returns **1536-dimensional** vectors and
is the cheapest of the current OpenAI embedding models, which makes it a good
default for learning and for most production retrieval.

Imports are grouped stdlib, then third-party, then LangChain.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and build the embeddings client
# ============================================================================
import tempfile

import numpy as np
from dotenv import load_dotenv

from langchain_classic.embeddings.cache import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_openai import OpenAIEmbeddings

# Load environment variables from .env file
load_dotenv()

# One client, reused by every section below.
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

print("✅ Environment variables loaded successfully!")
print(f"🤖 Embedding model: {embeddings_model.model}")

---

## 🧮 2. Basic Embeddings: One Text, One Vector

`embed_query()` takes a **single string** and returns a single list of floats. Use
it for the search query side of a retrieval system.

Three things are worth printing about any embedding: how many dimensions it has,
what the first few values look like, and its norm.

> **Key Insight**: the norm comes back at roughly `1.0` (expect values like
> `1.0002` or `0.9996`, since the API returns rounded floats). OpenAI embeddings
> are **L2-normalized**, so every vector sits on the unit sphere. That has a
> practical consequence used in section 4: for unit vectors, cosine similarity and
> the plain dot product agree.

In [ ]:
# ============================================================================
# BASIC EMBEDDINGS: Embed a single string with embed_query()
# ============================================================================
def basic_embeddings():
    """Embed one text and report the vector's shape, sample values and norm."""
    text = "What is Machine Learning?"

    single_embedding = embeddings_model.embed_query(text)

    print(f"📄 Text: {text}")
    print(f"🧮 Vector dimensions: {len(single_embedding)}")
    print(f"🔍 First 5 values: {single_embedding[:5]}")
    print(f"📐 Vector norm: {np.linalg.norm(single_embedding):.4f}")


# ---
basic_embeddings()

---

## 📦 3. Batch Embeddings: Many Texts, One Request

`embed_documents()` takes a **list of strings** and returns one vector per string.
Use it for the document side of a retrieval system.

Prefer it over calling `embed_query()` in a loop. One request for N texts means one
round trip instead of N, which is faster and cheaper at any real corpus size.

In [ ]:
# ============================================================================
# BATCH EMBEDDINGS: Embed a list of strings with embed_documents()
# ============================================================================
def batch_embeddings():
    """Embed several texts in a single API call and report each vector."""
    texts = [
        "What is Machine Learning?",
        "Explain the concept of overfitting in ML.",
        "How does a neural network work?",
    ]

    # One request for all three texts, not three separate requests.
    batch_embedding = embeddings_model.embed_documents(texts)

    print(f"📦 Embedded {len(batch_embedding)} texts in one call\n")
    for i, emb in enumerate(batch_embedding, start=1):
        print(f"📄 Text {i}: {texts[i - 1]}")
        print(f"   🧮 Dimensions: {len(emb)}")
        print(f"   🔍 First 5 values: {emb[:5]}")
        print(f"   📐 Norm: {np.linalg.norm(emb):.4f}\n")


# ---
batch_embeddings()

---

## 🔍 4. Similarity Search with Cosine Similarity

This is the core retrieval loop in miniature: embed the documents, embed the query,
score every document against the query, then sort.

### Key Concepts:
- **Cosine similarity**: the cosine of the angle between two vectors, ranging from
  `-1` (opposite) through `0` (unrelated) to `1` (identical direction). It compares
  *direction*, so it is unaffected by vector length.
- **Why not Euclidean distance**: length differences would distort the ranking.
  Cosine cares only about orientation, which is what "means the same thing" maps to.
- **Ranking**: sorting by score descending is exactly what a vector store's
  `similarity_search()` does for you, just at scale and with an index.

Watch the output ordering. The two programming documents should outrank the machine
learning ones, and `"Cats are popular pets"` should land last.

In [ ]:
# ============================================================================
# SIMILARITY SEARCH: Rank documents against a query by cosine similarity
# ============================================================================
def cosine_similarity(vec1, vec2):
    """Cosine of the angle between two vectors: dot product over the norms."""
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))


def similarity_search():
    """Embed a small corpus and a query, then rank the corpus by closeness."""
    docs = [
        "Python is a programming language",
        "JavaScript is used for web development",
        "Machine learning enables AI applications",
        "Deep learning uses neural networks",
        "Cats are popular pets",
    ]
    query = "What programming languages exist?"

    # Documents use embed_documents(), the query uses embed_query().
    doc_vectors = embeddings_model.embed_documents(docs)
    query_vector = embeddings_model.embed_query(query)

    similarities = [cosine_similarity(query_vector, dv) for dv in doc_vectors]

    # Highest score first — this is what a vector store returns as "top k".
    ranked_docs = sorted(zip(docs, similarities), key=lambda pair: pair[1], reverse=True)

    print(f"🔍 Query: {query}\n")
    print("📊 Ranked by similarity:")
    for doc, score in ranked_docs:
        print(f"   {score:.4f}  {doc}")


# ---
similarity_search()

---

## ⚡ 5. Caching Embeddings to Avoid Repeat API Calls

Embedding the same text twice costs the same money twice and returns the identical
vector. `CacheBackedEmbeddings` wraps any embeddings model and checks a key-value
store first, so only genuinely new text reaches the API.

How it works:

- **`underlying_embeddings`** is the real model, called only on a cache miss.
- **`document_embedding_cache`** is the store. `LocalFileStore` writes one file per
  vector; swap in Redis or another `ByteStore` for a shared cache.
- **`namespace`** keeps entries from different models apart. Always set it to
  something model-specific, or vectors from two models collide under one key.
- **`key_encoder`** hashes the text into a cache key. The default is `"sha1"`, which
  emits a warning because SHA-1 is not collision-resistant, so we pass `"sha256"`.

> **Note**: `query_embedding_cache` defaults to `False`, so only documents are
> cached. That is usually what you want, since queries are rarely repeated verbatim.

This demo uses a `TemporaryDirectory`, so the cache is discarded when the block
exits. Point `root_path` at a real directory to keep vectors between runs.

In [ ]:
# ============================================================================
# EMBEDDING CACHING: Wrap the model so repeat text skips the API
# ============================================================================
def embedding_caching():
    """Embed the same text twice and show the second call served from cache."""
    with tempfile.TemporaryDirectory() as tempdir:
        store = LocalFileStore(root_path=tempdir)

        cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
            underlying_embeddings=embeddings_model,  # called only on a cache miss
            document_embedding_cache=store,          # where vectors are persisted
            namespace="exercise",                    # keeps models from colliding
            key_encoder="sha256",                    # default "sha1" warns; sha256 does not
        )

        text = "What is Reinforcement Learning?"

        # --- First call: cache miss, so this hits the API
        print("🌐 First call (API):")
        vectors1 = cached_embeddings.embed_documents([text])
        print(f"   Embedded {len(vectors1)} document(s)")

        # --- Second call: cache hit, so no API request is made
        print("\n⚡ Second call (cache):")
        vectors2 = cached_embeddings.embed_documents([text])
        print(f"   Embedded {len(vectors2)} document(s)")

        # The cached vector must be byte-for-byte usable, not merely close.
        print(f"\n✅ Same vectors: {np.allclose(vectors1[0], vectors2[0])}")
        print(f"📁 Cache files written: {len(list(store.yield_keys()))}")


# ---
embedding_caching()

---

## 📝 Summary

In this notebook, we learned:

### 1. Two methods, two purposes
- **`embed_query(text)`**: one string in, one vector out. Use it for search queries.
- **`embed_documents(texts)`**: a list in, a list of vectors out, in one request.
  Use it for corpora, and never loop `embed_query()` in its place.

### 2. Reading a vector
- **Dimensions**: `text-embedding-3-small` returns 1536 floats per text.
- **Norm**: OpenAI embeddings are L2-normalized, so the norm is ≈ `1.0`. That is
  why cosine similarity and the dot product agree for these vectors.

### 3. Similarity search
- **Cosine similarity** compares direction, not magnitude, which is what makes it
  the right measure for semantic closeness.
- **Embed, score, sort** is the whole retrieval loop. A vector store adds an index
  so it stays fast over millions of documents instead of five.

### 4. Caching
- **`CacheBackedEmbeddings`** turns a repeat embedding into a local lookup.
- **Set `namespace`** per model, or two models will overwrite each other's keys.
- **Set `key_encoder="sha256"`** to avoid the SHA-1 collision warning.

### Next Steps
- `05_vector_stores.ipynb` — swap the hand-rolled ranking for a real indexed store
- `06_rag_pipeline.ipynb` — feed retrieved documents into an LLM for grounded answers